# 制造业 AI 演示 Notebook
## 不依赖 MES，如何从原始设备日志中提取“良率相关”洞察

### 业务背景（前因）
在很多工厂里，设备每天都会产生大量日志（上百列参数、秒级/分钟级时间序列），但现场团队经常遇到同一个问题：

- 数据很多，但难以快速回答“最近良率波动可能和哪些过程变量有关”。
- 完整 MES 项目虽然功能强，但建设周期长、跨系统打通复杂、导入成本高。
- 在没有良率标签（如 lot-level yield）的情况下，传统方法容易卡住。

### 本 Notebook 的目标（后果/价值）
本演示给客户一个“可快速落地”的路径：
1. 直接读取设备导出的 Excel/CSV 日志；
2. 构建一个干净的过程特征层；
3. 用轻量 AI（异常检测）找出“可疑工况时段”；
4. 对比正常 vs 异常段，近似推断潜在良率影响因子。

> **给客户的核心价值**：即使没有 MES、没有良率标签，也可以在几天内做出可行动的过程洞察。

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

## 1) 数据加载（Data Loading）
此步骤的关键点：
- 兼容 Excel / CSV；
- 尽量自动识别时间列并排序；
- 先建立“可复用的数据读取函数”，后续项目可直接复用。

**为什么这样做？**
现场数据常来自不同设备厂商，字段命名不统一。先把加载层做稳，后续分析才不会反复返工。

In [ ]:
def load_data(file_path, time_col_candidates=None):
    # 中文说明：统一入口读取设备日志，支持 Excel/CSV。
    # 前因：客户现场常见多种格式导出。
    # 后果：统一函数后，后续清洗/建模代码不需要改动。
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if time_col_candidates is None:
        # 中文说明：常见时间列候选名（可按客户现场再扩展）
        time_col_candidates = ["timestamp", "time", "datetime", "date_time", "record_time", "log_time"]

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    elif file_path.suffix.lower() == ".csv":
        df = pd.read_csv(file_path)
    else:
        raise ValueError("Unsupported file type. Use Excel (.xlsx/.xls) or CSV.")

    # 中文说明：尝试自动匹配时间列，并做时间解析+排序
    normalized_map = {c.lower().strip(): c for c in df.columns}
    selected_time_col = None
    for cand in time_col_candidates:
        if cand in normalized_map:
            selected_time_col = normalized_map[cand]
            break

    if selected_time_col is not None:
        df[selected_time_col] = pd.to_datetime(df[selected_time_col], errors="coerce")
        df = df.sort_values(selected_time_col).reset_index(drop=True)
        print(f"Timestamp column parsed: {selected_time_col}")
    else:
        print("No timestamp column detected from candidates.")

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df

In [ ]:
# --- Update this path to your actual Excel log file ---
# Example: data_path = "data/equipment_log.xlsx"
# For quick demo fallback, we can also read the provided CSV in this repo.

data_path = "data/aami_yield_demo.csv"
df_raw = load_data(data_path)

display(df_raw.head(3))
display(df_raw.describe(include="all").transpose().head(10))

## 2) 数据理解（EDA）
工程师最先看的通常是关键趋势图：
- **Speed（速度）**：反映节拍与产能状态；
- **pH**：反映化学条件稳定性；
- **Pressure（压力）**：反映设备/过程应力状态。

**业务含义**：
这一步不是“炫技可视化”，而是模拟现场工程师做首轮排查的方式。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def _find_column(df, keywords):
    cols_lower = {c.lower(): c for c in df.columns}
    for c_low, c_org in cols_lower.items():
        if all(k.lower() in c_low for k in keywords):
            return c_org
    return None


def plot_key_variables(df):
    speed_col = _find_column(df, ["speed"]) or _find_column(df, ["line", "speed"])
    ph_col = _find_column(df, ["ph"])
    pressure_col = _find_column(df, ["pressure"])

    key_cols = [c for c in [speed_col, ph_col, pressure_col] if c is not None]
    if not key_cols:
        raise ValueError("Could not identify key variables (speed/pH/pressure).")

    n = len(key_cols)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3.5 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, key_cols):
        ax.plot(df.index, pd.to_numeric(df[col], errors="coerce"), linewidth=1.1)
        ax.set_title(f"Trend: {col}")
        ax.set_ylabel(col)

    axes[-1].set_xlabel("Record Index (or Time Order)")
    plt.tight_layout()
    plt.show()

    print("Detected columns:")
    print({"speed": speed_col, "pH": ph_col, "pressure": pressure_col})
    return {"speed": speed_col, "pH": ph_col, "pressure": pressure_col}

In [ ]:
key_map = plot_key_variables(df_raw)

## 3) 特征选择与清洗层（Feature Selection & Clean Layer）
原始日志可能有 ~173 列，但并非每一列都与过程稳定性高度相关。

这里我们做三件事：
1. 选出约 10 个高价值过程变量；
2. 列名标准化（snake_case），降低跨项目维护成本；
3. 缺失值处理，得到稳定可用的 `df_clean`。

**前因后果**：
如果不先做 clean layer，后续异常检测结果会被脏数据放大，客户会质疑结论可信度。

In [ ]:
def clean_data(df, selected_cols=None):
    # 中文说明：构建 df_clean（分析中间层）
    # 目标：让后续特征工程和异常检测都基于“可解释、可复用、可上线”的数据。
    df = df.copy()

    if selected_cols is None:
        # 中文说明：按制造场景常见重要变量做优先匹配
        priority_patterns = [
            ["speed"], ["ph"], ["pressure"], ["temp"], ["temperature"],
            ["flow"], ["current"], ["voltage"], ["torque"], ["vibration"], ["rpm"]
        ]

        selected_cols = []
        for p in priority_patterns:
            col = _find_column(df, p)
            if col and col not in selected_cols:
                selected_cols.append(col)
            if len(selected_cols) >= 10:
                break

    if not selected_cols:
        raise ValueError("No columns selected for cleaning.")

    df_clean = df[selected_cols].copy()

    # 中文说明：字段标准化，避免中文/符号/空格导致脚本不稳定
    def normalize_col(c):
        c = c.strip().lower()
        c = re.sub(r"[^a-z0-9]+", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        return c

    rename_map = {c: normalize_col(c) for c in df_clean.columns}
    df_clean = df_clean.rename(columns=rename_map)

    # 中文说明：尽量转为数值列，无法转换的记为 NaN
    for c in df_clean.columns:
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

    # 中文说明：缺失值处理策略
    # 1) ffill/bfill 保留时间连续性
    # 2) 若仍有缺失，用中位数兜底（抗异常值能力比均值更好）
    df_clean = df_clean.ffill().bfill()
    for c in df_clean.columns:
        if df_clean[c].isna().any():
            df_clean[c] = df_clean[c].fillna(df_clean[c].median())

    print(f"df_clean shape: {df_clean.shape}")
    print("Columns:", list(df_clean.columns))
    return df_clean

In [ ]:
df_clean = clean_data(df_raw)
display(df_clean.head())

## 4) 特征工程（Feature Engineering）
我们把原始点位转换成“更接近工艺行为”的特征：

- `speed_diff`：速度变化强度（识别节拍波动）；
- `pH_range`：滚动窗口内 pH 振幅（识别化学稳定性风险）；
- `pressure_variation`：滚动压力波动（识别设备/工艺应力波动）。

**业务意义**：
客户真正关心的是“过程是否稳定”，不是单个点位瞬时值本身。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def engineer_features(df_clean):
    df_feat = df_clean.copy()

    speed_col = _find_column(df_feat, ["speed"]) or _find_column(df_feat, ["rpm"])
    ph_col = _find_column(df_feat, ["ph"])
    pressure_col = _find_column(df_feat, ["pressure"])

    if speed_col is None:
        raise ValueError("No speed-like column found for speed_diff.")
    if ph_col is None:
        raise ValueError("No pH column found for pH_range.")
    if pressure_col is None:
        raise ValueError("No pressure column found for pressure_variation.")

    df_feat["speed_diff"] = df_feat[speed_col].diff().abs().fillna(0)

    roll_window = 12
    ph_roll_max = df_feat[ph_col].rolling(roll_window, min_periods=1).max()
    ph_roll_min = df_feat[ph_col].rolling(roll_window, min_periods=1).min()
    df_feat["pH_range"] = ph_roll_max - ph_roll_min

    df_feat["pressure_variation"] = (
        df_feat[pressure_col].rolling(roll_window, min_periods=2).std().fillna(0)
    )

    return df_feat

In [ ]:
df_feat = engineer_features(df_clean)
display(df_feat[["speed_diff", "pH_range", "pressure_variation"]].head(15))

## 5) 异常检测（Anomaly Detection）
在没有良率标签时，我们先识别“异常工况时段”。

方法：
- 对每个数值特征做 z-score；
- 聚合为 `anomaly_score`；
- 用分位数阈值得到 `anomaly_flag`。

**对客户的解释话术**：
这不是替代 SPC 规则，而是提供一层“无监督 AI 雷达”，帮助快速定位需要优先排查的时间段。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def detect_anomaly(df_feat, threshold_quantile=0.97):
    df_out = df_feat.copy()
    num_cols = df_out.select_dtypes(include=[np.number]).columns.tolist()

    z_dict = {}
    for c in num_cols:
        mean = df_out[c].mean()
        std = df_out[c].std(ddof=0)
        if std == 0 or np.isnan(std):
            z_dict[c] = np.zeros(len(df_out))
        else:
            z_dict[c] = (df_out[c] - mean) / std

    z_df = pd.DataFrame(z_dict, index=df_out.index)
    df_out["anomaly_score"] = z_df.abs().mean(axis=1)

    cutoff = df_out["anomaly_score"].quantile(threshold_quantile)
    df_out["anomaly_flag"] = (df_out["anomaly_score"] >= cutoff).astype(int)

    print(f"Anomaly cutoff (q={threshold_quantile:.2f}): {cutoff:.3f}")
    print(df_out["anomaly_flag"].value_counts().rename({0: "NORMAL", 1: "ABNORMAL"}))
    return df_out

In [ ]:
df_anom = detect_anomaly(df_feat, threshold_quantile=0.97)

def plot_anomalies(df, y_col="anomaly_score"):
    plt.figure(figsize=(14, 4))
    plt.plot(df.index, df[y_col], label=y_col, linewidth=1.2)

    abnormal_idx = df.index[df["anomaly_flag"] == 1]
    plt.scatter(abnormal_idx, df.loc[abnormal_idx, y_col], color="red", s=18, label="Anomaly", alpha=0.8)
    plt.title("Anomaly Detection Result")
    plt.xlabel("Record Index")
    plt.ylabel(y_col)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_anomalies(df_anom)

## 6) 分段对比（核心洞察）
将数据分为两段：
- NORMAL（正常段）
- ABNORMAL（异常段）

然后对比均值并按差异排序，得到“最值得关注的变量”。

**为什么这一步接近良率分析？**
虽然没有直接 yield 标签，但异常段常常对应过程偏离。比较两段差异，可以近似识别“潜在影响良率”的工艺因子。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def compare_segments(df_anom, top_n=10):
    num_cols = df_anom.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ["anomaly_flag"]]

    grp = df_anom.groupby("anomaly_flag")[num_cols].mean().T
    grp.columns = ["normal_mean", "abnormal_mean"]

    grp["abs_diff"] = (grp["abnormal_mean"] - grp["normal_mean"]).abs()
    grp["pct_diff_vs_normal"] = np.where(
        grp["normal_mean"].abs() > 1e-12,
        (grp["abnormal_mean"] - grp["normal_mean"]) / grp["normal_mean"] * 100,
        np.nan,
    )

    ranked = grp.sort_values("abs_diff", ascending=False)
    return ranked.head(top_n), ranked

In [ ]:
top_diff, full_diff = compare_segments(df_anom, top_n=10)
print("Top differences between NORMAL and ABNORMAL segments:")
display(top_diff)

## 7) AI 文本洞察生成
我们把统计差异自动转成现场可读的结论：
- 哪些变量变化最大；
- 变化方向是什么；
- 建议先做哪些工程检查。

可选：调用本地 Ollama 生成更自然的客户报告文案。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def generate_insight(top_diff, max_items=5):
    lines = []
    lines.append("AI Insight Summary (Normal vs Abnormal)")
    lines.append("-" * 50)

    focus = top_diff.head(max_items).reset_index().rename(columns={"index": "feature"})
    for i, row in focus.iterrows():
        direction = "higher" if row["abnormal_mean"] > row["normal_mean"] else "lower"
        lines.append(
            f"{i+1}. {row['feature']}: abnormal is {direction} than normal "
            f"(normal={row['normal_mean']:.3f}, abnormal={row['abnormal_mean']:.3f}, "
            f"delta={row['pct_diff_vs_normal']:.1f}%)."
        )

    lines.append("\nRecommended next checks:")
    lines.append("- Verify control limits and setpoint drift on top-shifted variables.")
    lines.append("- Cross-check abnormal windows with maintenance / lot change events.")
    lines.append("- Use these variables as priority candidates for root-cause and DOE planning.")
    return "\n".join(lines)


def generate_insight_with_ollama(prompt, model="llama3.1", url="http://localhost:11434/api/generate"):
    """
    Optional helper: call a local Ollama model for richer narrative output.
    Requires `ollama serve` and a pulled model.
    """
    import requests

    payload = {"model": model, "prompt": prompt, "stream": False}
    resp = requests.post(url, json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json().get("response", "")

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
insight_text = generate_insight(top_diff, max_items=5)
print(insight_text)

# Optional: local LLM enrichment (uncomment if Ollama is running)
# llm_prompt = f"""
# You are a manufacturing process expert.
# Convert the following analysis into a concise customer-facing insight summary.
#

# {insight_text}
# """
# print(generate_insight_with_ollama(llm_prompt, model="llama3.1"))

## 8) 最终结论（给管理层）
- 这套方法可替代 MES 分析中的一部分“早期诊断”能力；
- 不需要良率标签也能先做过程风险定位；
- 适合快速 PoC，再逐步接入更多系统（MES/QMS/维保事件）。

**落地收益**：
上线快、投入轻、见效早，适合中国制造客户先小步试点再规模复制。

## 9) Demo Talking Points（客户演示讲稿）
1. **先解决 80% 的问题，再谈大平台**：不用等 MES 全部打通，先从设备日志直接产出洞察。  
2. **没有良率标签也能做**：通过异常段对比，先找出高风险变量，提升排查效率。  
3. **结果可解释，工程师听得懂**：趋势图 + 差异表 + 中文洞察建议。  
4. **技术栈轻量，IT 阻力小**：pandas/numpy/matplotlib 即可启动。  
5. **可平滑升级**：后续接入 yield、lot、维保记录后，模型效果会持续增强。

---

### 给中国客户的收尾话术（可直接口播）
> 我们这套方法不是替代 MES，而是让您在 MES 完整建设前，就先拿到可行动的工艺洞察。  
> 先快速看到异常、定位重点变量、缩小排查范围，再决定下一步系统化投资，风险更低、回报更快。